# Register an Azure ML Environment

Define a versioned environment from a base image and checked-in Conda file, register it, and retrieve it for verification.

**Source:** Adapted from [Azure/azureml-examples environment.ipynb](https://github.com/Azure/azureml-examples/blob/7dbe9a3ddfc4a920a9de82eaa4af7eaf118841d8/sdk/python/assets/environment/environment.ipynb), MIT License.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import MLClient
from azure.ai.ml.entities import Environment
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
ENVIRONMENT_NAME = os.environ["WORKSHOP_ENVIRONMENT_NAME"]
ENVIRONMENT_VERSION = os.environ["WORKSHOP_ENVIRONMENT_VERSION"]
REGISTER = os.getenv("REGISTER_FOUNDATION_ENVIRONMENT", "false").lower() in {"1", "true", "yes"}

In [ ]:
conda_file = WORKSHOP_ROOT / "environment/foundations/conda.yaml"
environment_definition = Environment(
    name=ENVIRONMENT_NAME,
    version=ENVIRONMENT_VERSION,
    image="mcr.microsoft.com/azureml/minimal-py310-inference:latest",
    conda_file=str(conda_file),
    description="Small workshop environment for command and endpoint demonstrations",
    tags={"workshop": "azureml-h2o", "purpose": "foundations"},
)

if REGISTER:
    registered_environment = ml_client.environments.create_or_update(environment_definition)
    verified_environment = ml_client.environments.get(ENVIRONMENT_NAME, ENVIRONMENT_VERSION)
    assert verified_environment.name == ENVIRONMENT_NAME
    assert str(verified_environment.version) == ENVIRONMENT_VERSION
    print(f"Registered environment: {verified_environment.name}:{verified_environment.version}")
else:
    print(f"Prepared environment definition: {ENVIRONMENT_NAME}:{ENVIRONMENT_VERSION}")
    print("Registration disabled. Set REGISTER_FOUNDATION_ENVIRONMENT=true in workshop/.env.")

## Expected Result

The checked-in Conda specification defines a named, immutable Azure ML environment that can be retrieved by name and version.

Next: `04_register_model.ipynb`.